# Compute selectivity & specificity index 

## Import recordings

Load packages

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import scipy
from scipy import signal
import os
from ephyviewer import mkQApp, MainViewer, TraceViewer
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, InMemoryAnalogSignalSource, EpochEncoder,TimeFreqViewer,AnalogSignalSourceWithScatter, EpochEncoder_ABC
from matplotlib import cm
from matplotlib.colors import to_hex
import re
import mne
import pandas as pd
import numpy as np
import csv
from collections import defaultdict
import ast
import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import string
import pickle
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
%matplotlib widget

Choose a sp_spikes_spmeeg_XXX_repaired_montage.mat file

In [ ]:
dpath = "//10.69.168.1/crnldata/forgetting/Aurelie/EpiKid/SpikeDet/"
fc1 = FileChooser(dpath,select_default=True, show_only_dirs = False, title = "<b>Choose a .mat file with spike detection results </b>", layout=widgets.Layout(width='100%'))
fc1.filter_pattern = '*.mat'
display(fc1)
def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return 
fc1.register_callback(update_my_folder)

Load cleaned edf file

In [ ]:
folder_base= Path(dpath)
file=('_').join(folder_base.name.split("_")[3:])[:-4]
data = mne.io.read_raw_edf(f'{folder_base.parent.parent.parent}/CleanedData/{file}.edf', preload=False)
raw_data = data.get_data() 

info = data.info
channels = data.ch_names
timestamps = data.times
duration = data.duration
samplerate = data.info.get('sfreq')  # in Hz
data.info.get('subject_info')
lowpass=data.info.get('lowpass')
highpass=data.info.get('highpass')

combined = raw_data.T
print(f'{round(np.shape(raw_data)[1]/samplerate/60/60,3)}h recording if sampled at {samplerate}Hz ({duration} seconds)')
print(f'{np.shape(raw_data)[0]} channels found: {channels}')

Zscore & filter signals

In [ ]:
montage_eeg=[]
montage_name=[]
for i, ch in enumerate(channels):
    if ch.count('-')==0:
        ch1 = ch
        eeg=combined[:,i]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        #montage_eeg = (eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, (eeg[:,np.newaxis]), axis=1)
        montage_name.append(ch)
    else:        
        ch1 = ch
        eeg=combined[:,i]
        nyq = samplerate / 2
        f_lowcut = 0.5
        f_hicut = 50.0 #70
        Wn = [f_lowcut/nyq, f_hicut/nyq]
        sos = signal.butter(6, Wn, btype='band', output='sos')
        eeg = signal.sosfiltfilt(sos, eeg)
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        #montage_eeg = (eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, (eeg[:,np.newaxis]), axis=1)
        montage_name.append(ch)
montage_name_eeg=montage_name[:-2]

Load automatically detected spikes, manually detected spikes and sleep scoring file

In [ ]:
matsp = scipy.io.loadmat(dpath, squeeze_me=True, struct_as_record=False)
sp_array = matsp['sp']
autoDetect_spikes = pd.concat([
    pd.DataFrame({
        'time': np.round(sp_array[i].GenDet.Epoch / samplerate, 2),
        'duration': 0.01,
        'label': channel
    })
    for i, channel in enumerate(montage_name_eeg)], ignore_index=True).sort_values('time').reset_index(drop=True)

SleepScoring_filename = Path(folder_base).parent.parent.parent / "CleanedData" / f"EphyViewer_Scoring_{'_'.join(file.split('_')[:2])}.csv"
SleepScoring = pd.read_csv(SleepScoring_filename)

SpikesScoring_filename = Path(folder_base).parent.parent.parent / "CleanedData" / f"EphyViewer_Spikes_{'_'.join(file.split('_')[:2])}.csv"
SpikesScoring = pd.read_csv(SpikesScoring_filename)

Identify sleep stages of each spikes and truncate file with auto detection to match manual one

In [ ]:
def find_stages_vectorized(time_values, df):
    df = df.copy()
    df['end_time'] = df['time'] + df['duration']
    # Convert to numpy arrays for faster operations
    times = time_values.values if hasattr(time_values, 'values') else np.array(time_values)
    df_times = df['time'].values
    df_end_times = df['end_time'].values
    df_labels = df['label'].values
    in_interval = (times[:, np.newaxis] >= df_times) & (times[:, np.newaxis] < df_end_times)
    matches = np.argmax(in_interval, axis=1)
    has_match = in_interval.any(axis=1)
    stages = np.where(has_match, df_labels[matches], None)
    return stages

# Identify sleep stages for each spike in manually detected spikes and autoDetect_spikes
times_sec = SpikesScoring['time'].values
stages = find_stages_vectorized(times_sec, SleepScoring)
SpikesScoring['stage'] = stages

times_sec = autoDetect_spikes['time'].values
stages = find_stages_vectorized(times_sec, SleepScoring)
autoDetect_spikes['stage'] = stages

full_dfauto=pd.DataFrame()
# Truncate autoDetect_spikes to match manually detected spikes for each stage
for stage in SpikesScoring['stage'].unique():
    SpikesScoring_bis= SpikesScoring[SpikesScoring['stage']==stage]
    start_time = SpikesScoring_bis['time'].min()-0.3
    end_time = SpikesScoring_bis['time'].max()+0.3
    dfauto= f'autoDetect_spikes_{stage.replace(" ", "_")}'
    globals()[dfauto] = autoDetect_spikes[(autoDetect_spikes['time'] >= start_time) & (autoDetect_spikes['time'] <= end_time)]
    full_dfauto = pd.concat([full_dfauto, globals()[dfauto]], ignore_index=True)
    dfmanu= f'SpikesScoring_{stage.replace(" ", "_")}'
    globals()[dfmanu] = SpikesScoring_bis

Compute specificity & precision 

In [ ]:
def compute_detection_metrics(auto_df, manual_df, tolerance=0.1):   
    auto_times = auto_df['time'].values
    manual_times = manual_df['time'].values
    if len(auto_times) == 0 and len(manual_times) == 0:
        return {'sensitivity': 100.0,'specificity': 100.0}
    elif len(auto_times) == 0 or len(manual_times) == 0:
        return {'sensitivity': 0.0,'specificity': 0.0}
    distances = np.abs(auto_times[:, np.newaxis] - manual_times[np.newaxis, :])
    within_tolerance = distances <= tolerance
    manual_matched = np.zeros(len(manual_times), dtype=bool)
    auto_matched = np.zeros(len(auto_times), dtype=bool)
    auto_idx, manual_idx = np.where(within_tolerance)
    if len(auto_idx) > 0:
        dists = distances[auto_idx, manual_idx]
        sort_order = np.argsort(dists)
        for idx in sort_order:
            a_idx = auto_idx[idx]
            m_idx = manual_idx[idx]
            if not auto_matched[a_idx] and not manual_matched[m_idx]:
                auto_matched[a_idx] = True
                manual_matched[m_idx] = True
    true_positives = np.sum(auto_matched)
    false_positives = np.sum(~auto_matched)
    false_negatives = np.sum(~manual_matched)
    sensitivity = (true_positives / (true_positives + false_negatives) * 100) if (true_positives + false_negatives) > 0 else 0
    precision = (true_positives / (true_positives + false_positives) * 100) if (true_positives + false_positives) > 0 else 0
    return {'sensitivity': round(sensitivity, 1),'specificity': round(precision, 1)}

results = []
for stage in SpikesScoring['stage'].unique():
    dfauto = globals()[f'autoDetect_spikes_{stage.replace(" ", "_")}']
    dfmanu = globals()[f'SpikesScoring_{stage.replace(" ", "_")}']
    for ch in montage_name_eeg:
        result = compute_detection_metrics(
            dfauto[dfauto['label'] == ch],
            dfmanu[dfmanu['label'] == ch],
            tolerance=.2)
        result.update({'stage': stage, 'channel': ch})
        results.append(result)
df_results = pd.DataFrame(results)
result_per_st_per_ch = {f"{row['stage']}_{row['channel']}": row.to_dict() for _, row in df_results.iterrows()}
result_per_st = df_results.groupby('stage')[['sensitivity', 'specificity']].mean().round(1).to_dict('index')
result_per_ch = df_results.groupby('channel')[['sensitivity', 'specificity']].mean().round(1).to_dict('index')
result_per_ch = {k: result_per_ch[k] for k in montage_name_eeg if k in result_per_ch}

result_per_ch

Display auto & manually detected spikes

In [ ]:
scatter_channels = defaultdict(list)
scatter_indexes = defaultdict(list)
color_dict = {}
r=0
for i, channel in enumerate(montage_name_eeg):
    full_dfauto_ch= full_dfauto[full_dfauto['label']==channel]
    if not full_dfauto_ch.empty:
        scatter_indexes[r] = pd.Series(np.round(full_dfauto_ch['time'].values*samplerate, 0).astype(int))
        scatter_channels[r] = [i]
        color_dict[r] = "#0400FF80"


    SpikesScoring_ch= SpikesScoring[SpikesScoring['label']==channel]
    if not SpikesScoring_ch.empty:
        scatter_indexes[r+1] = pd.Series(np.round(SpikesScoring_ch['time'].values*samplerate, 0).astype(int))
        scatter_channels[r+1] = [i]
        color_dict[r+1] = "#FF000080"
        r+=2
    else:
        r+=1

scatter_channels = dict(sorted(scatter_channels.items()))
scatter_indexes = dict(sorted(scatter_indexes.items()))
color_dict = dict(sorted(color_dict.items()))

In [ ]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
winlen = 30 # default window length in sec

SleepScoring_filename = Path(folder_base).parent.parent.parent / "CleanedData" / f"EphyViewer_Scoring_{'_'.join(file.split('_')[:2])}.csv"
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
source = AnalogSignalSourceWithScatter(montage_eeg, samplerate, t_start, scatter_indexes, scatter_channels,  scatter_colors = color_dict, channel_names=montage_name)
view1 = TraceViewer(source=source)
#scatter_colors = color_dict,
view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = '#FFFFFF'
view1.params['label_fill_color'] = '#FFFFFF'

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()

#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#0000ff"
view2.params['background_color'] = '#FFFFFF'
view2.params['label_fill_color'] = '#FFFFFF'
view2.params['view_mode'] = 'flat'
view2.controls.hide()


win.add_view(view1)
win.add_view(view2)
win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

# press '1', '2', '3', '4' etc, to encode state.
# or toggle 'Time range selector' and then use 'Insert within range'
